In [17]:
import os
os.environ["PYSPARK_SUBMIT_ARGS"] = (
    "--packages org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0 "
    "pyspark-shell"
)

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json, to_timestamp
from pyspark.sql.types import *

spark = SparkSession.builder \
    .appName("FinancialConsumer") \
    .master("spark://spark-master:7077") \
    .config("spark.hadoop.fs.defaultFS", "hdfs://namenode:9000") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print("✅ Spark ready!")

✅ Spark ready!


In [18]:
schema = StructType([
    StructField("symbol",     StringType(),  True),
    StructField("timestamp",  StringType(),  True),
    StructField("price",      DoubleType(),  True),
    StructField("volume",     LongType(),    True),
    StructField("market_cap", DoubleType(),  True),
])

df = spark.read \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "kafka:29092") \
    .option("subscribe", "financial-data") \
    .option("startingOffsets", "earliest") \
    .load()

parsed = df \
    .selectExpr("CAST(value AS STRING) as json_value") \
    .withColumn("data", from_json(col("json_value"), schema)) \
    .select(
        col("data.symbol"),
        col("data.timestamp").alias("event_time"),
        col("data.price"),
        col("data.volume"),
        col("data.market_cap"),
    )

parsed.show()
print("✅ Data read from Kafka!")

ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "/usr/local/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.11/socket.py", line 706, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt


KeyboardInterrupt: 

In [19]:
spark.stop()

In [20]:
spark = SparkSession.builder \
    .appName("FinancialConsumer") \
    .master("local[*]") \
    .config("spark.hadoop.fs.defaultFS", "hdfs://namenode:9000") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print("✅ Spark ready in local mode!")

✅ Spark ready in local mode!


In [21]:
df = spark.read \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "kafka:29092") \
    .option("subscribe", "financial-data") \
    .option("startingOffsets", "earliest") \
    .load()

print("✅ Read from Kafka!")
print("Raw rows:", df.count())

✅ Read from Kafka!
Raw rows: 5


In [22]:
parsed = df \
    .selectExpr("CAST(value AS STRING) as json_value") \
    .withColumn("data", from_json(col("json_value"), schema)) \
    .select(
        col("data.symbol"),
        col("data.timestamp").alias("event_time"),
        col("data.price"),
        col("data.volume"),
        col("data.market_cap"),
    )

parsed.show()

+------+--------------------+------+--------+--------------------+
|symbol|          event_time| price|  volume|          market_cap|
+------+--------------------+------+--------+--------------------+
|  AAPL|2026-04-26T23:08:...|271.06|38124500|3.979469772557373E12|
|  MSFT|2026-04-26T23:08:...|424.62|27413900|3.155936163999813...|
| GOOGL|2026-04-26T23:08:...| 344.4|26400000|4.166206726165771...|
|  AMZN|2026-04-26T23:08:...|263.99|53695200|2.839014827396019...|
|  TSLA|2026-04-26T23:08:...| 376.3|62753700|1.413278846811061...|
+------+--------------------+------+--------+--------------------+



In [23]:
parsed.write \
    .mode("overwrite") \
    .parquet("hdfs://namenode:9000/user/bigdata/financial")

print("✅ Data saved to HDFS!")

✅ Data saved to HDFS!
